# ADK 2.x Routing Workflow

This notebook demonstrates how to build a conditional routing workflow using `google.adk`. 

The workflow uses an LLM `Agent` to classify an incoming user message, passes that classification to a Python `router` node, and then dynamically directs the execution flow to one or more specific destination nodes based on the intent.

## 2. Imports & Setup

First, let's ensure we are using the version of the Google Agent Development Kit > 2.5

In [1]:
import google.adk

print(google.adk.__version__)

2.5.0


Import the required ADK components.

In [13]:
from google.adk import Agent
from google.adk import Workflow
from google.adk import Event
from pydantic import BaseModel
import os
# Define the model to be used by our LLM Agent
MODEL = "gemini-2.5-flash"

In [14]:
LOCATION = "us-central1"
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"  # Use Agent Platform API

In [18]:
%%bash
mkdir -p adk2_agents
echo > adk2_agents/.env "GOOGLE_CLOUD_LOCATION=$GOOGLE_CLOUD_LOCATION
GOOGLE_GENAI_USE_VERTEXAI=$GOOGLE_GENAI_USE_VERTEXAI
"

## 3. Define the LLM Classifier Agent
This node uses Gemini to analyze the text and output a strictly formatted string. If it detects multiple intents, it will return a comma-separated list.

In [19]:
process_message = Agent(
    name="process_message",
    model=MODEL,
    instruction="""Classify user message into either "BUG", "CUSTOMER_SUPPORT",
     or "LOGISTICS". If you think a message applies to more than one category,
     reply with a comma separated list of categories.
  """,
    output_schema=str,
)

## 4. Define the Router Node
The router node takes the text output from the LLM, splits it into a list, and emits an `Event(route=...)`. 
In ADK 2.x, setting the `route` property on an Event tells the workflow engine which path(s) to follow next.

In [20]:
def router(node_input: str):
    # Split the comma-separated string from the LLM
    routes = node_input.split(",")
    # Clean up any whitespace
    routes = [route.strip() for route in routes]
    
    # Emitting an Event with 'route' dictates the next edge(s) to traverse
    return Event(route=routes)

## 5. Define Destination Nodes
These functions act as the endpoints for our different categories. If a message contains multiple categories, the workflow will trigger the corresponding nodes in parallel.

In [21]:
def response_1_bug():
    return Event(message="Handling bug...")

def response_2_support():
    return Event(message="Handling customer support...")

def response_3_logistics():
    return Event(message="Handling logistics...")

## 6. Construct the Workflow
We tie it all together in the `Workflow`. 
Notice the dictionary in the second edge: `(router, { "ROUTE_NAME": destination_node })`. This syntax automatically maps the strings yielded by `Event(route=...)` to specific nodes.

In [22]:
root_agent = Workflow(
    name="routing_workflow",
    edges=[
        # 1. Start execution at the LLM classifier, then pass output to router
        ("START", process_message, router),
        
        # 2. Map the routes emitted by the router to the destination nodes
        (router,
            {
                "BUG": response_1_bug,
                "CUSTOMER_SUPPORT": response_2_support,
                "LOGISTICS": response_3_logistics,
            }
        )
    ],
)

## 7. Test the Workflow
Finally, we use the `Runner` and `InMemorySessionService` to test the logic asynchronously. We will feed it a message that should trigger both the **BUG** and **LOGISTICS** routes simultaneously.

In [24]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

app_name = "routing_app"
user_id = "user_123"
session_id = "session_001"

# 1. Initialize session storage
session_service = InMemorySessionService()

# 2. Await the session creation
session = await session_service.create_session(
    app_name=app_name, 
    user_id=user_id,
    session_id=session_id
)

# 3. Create the Runner
runner = Runner(
    agent=root_agent,
    app_name=app_name,
    session_service=session_service
)

### Define a test message that crosses multiple intents

In [25]:
test_input = "I can't track my shipment, and the mobile app keeps crashing!"
input_message = Content(role="user", parts=[Part(text=test_input)])

print(f"Testing input: '{test_input}'\n")
print("Starting workflow execution...\n")

# 5. Execute asynchronously and print node messages
async for event in runner.run_async(
    user_id=session.user_id,
    session_id=session.id,
    new_message=input_message
):
    if event.message:
        print(f"> {event.message}")

Testing input: 'I can't track my shipment, and the mobile app keeps crashing!'

Starting workflow execution...

> parts=[Part(
  text='"LOGISTICS,BUG"'
)] role='model'
> parts=[Part(
  text='Handling bug...'
)] role='user'
> parts=[Part(
  text='Handling logistics...'
)] role='user'


Copyright 2026 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License